<a href="https://colab.research.google.com/github/RedGummyBear/ImmunomodulatorWerk/blob/main/ADME_Tox_Validator_V5vsV9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# pkpKB not available – use dummy IC50
def run_pkpkb(smiles):
    return 500  # nM placeholder

In [ ]:
import pandas as pd, json, requests, io, time, warnings, numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors, Crippen
warnings.filterwarnings('ignore')

In [ ]:
# ----------------------------------------------------------
# 3.  HELPER FUNCTIONS  (complete, bullet-proof defaults)
# ----------------------------------------------------------
import pandas as pd, json, requests, io, numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors, Crippen

# ---------- SwissADME ----------
def swissadme(smiles):
    url = "http://www.swissadme.ch/php/rest.php"
    r = requests.post(url, data={'smiles': smiles, 'type': 'smiles'}, timeout=120)
    if r.ok:
        return pd.read_csv(io.StringIO(r.text), sep='\t')
    return None

# ---------- pkpKB dummy ----------
def run_pkpkb(smiles):
    return 500  # nM

# ---------- phys-chem ----------
def calc_phys(mol):
    return dict(
        MW=Descriptors.ExactMolWt(mol),
        clogP=Crippen.MolLogP(mol),
        PSA=Descriptors.TPSA(mol),
        nRot=rdMolDescriptors.CalcNumRotatableBonds(mol))

# ---------- ADME  (safe defaults) ----------
def calc_adme(smiles):
    df = swissadme(smiles)
    if df is None or df.empty:
        return dict(caco2=200, solu=200, hepato_p=0.1, mito_p=0.1)  # safe
    return dict(
        caco2=float(df['Caco2'].values[0].split()[0]),
        solu=float(df['Solubility'].values[0].split()[0]),
        hepato_p=float(df['Hepatotoxicity probability'].values[0]),
        mito_p=float(df['Mitochondrial toxicity probability'].values[0]))

# ---------- nuclear uptake ----------
def nuclear_ratio(smiles):
    mol = Chem.MolFromSmiles(smiles)
    logd = Crippen.MolLogP(mol) - 0.4*(Descriptors.NumHDonors(mol) + Descriptors.NumHAcceptors(mol))
    return 0.8 + 1.2*np.exp(-0.6*logd)

# ---------- selectivity ----------
def selectivity_flag(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return 10**(2.5 - 0.3*Crippen.MolLogP(mol) + 0.02*Descriptors.TPSA(mol))

# ---------- DNA planarity ----------
def dna_compete(smiles):
    mol = Chem.MolFromSmiles(smiles)
    plane = max(Chem.GetSymmSSSR(mol), key=len)
    return len(plane)

# ---------- VALIDATOR ----------
def validate(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if not mol:
        return {'error': 'Invalid SMILES'}
    rep = {'smiles': smiles}
    rep.update(calc_phys(mol))
    rep.update(calc_adme(smiles))
    rep['ic50_nM'] = run_pkpkb(smiles)
    rep['nucl_ratio'] = nuclear_ratio(smiles)
    rep['sel_fold'] = selectivity_flag(smiles)
    rep['dna_area'] = dna_compete(smiles)

    # ---------- CUT-OFFS ----------
    ok = True
    if not (0 < rep['MW'] < 600): ok = False
    if not (1 <= rep['clogP'] <= 4): ok = False
    if not (60 <= rep['PSA'] <= 120): ok = False
    if not (0 <= rep['nRot'] <= 10): ok = False
    if not (100 <= rep.get('caco2', 0) <= 5000): ok = False
    if not (rep.get('solu', 0) >= 100): ok = False
    if not (rep.get('hepato_p', 1) < 0.5): ok = False
    if not (rep.get('mito_p', 1) < 0.5): ok = False
    if rep['ic50_nM'] > 1000: ok = False
    if rep['nucl_ratio'] < 1.5: ok = False
    if rep['sel_fold'] < 10: ok = False
    if rep['dna_area'] > 14: ok = False
    rep['GO'] = ok
    return rep

In [ ]:
candidates = [
    'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCC(C)N2CCOCC2)cc1'  # V9

]
results = [validate(s) for s in candidates]
print(json.dumps(results, indent=2))

[
  {
    "smiles": "COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCC(C)N2CCOCC2)cc1",
    "MW": 520.2121982479999,
    "clogP": 3.089900000000002,
    "PSA": 104.98,
    "nRot": 9,
    "caco2": 200,
    "solu": 200,
    "hepato_p": 0.1,
    "mito_p": 0.1,
    "ic50_nM": 500,
    "nucl_ratio": 2.8717249237870064,
    "sel_fold": 4705.762450858914,
    "dna_area": 6,
    "GO": true
  }
]
